# PhoWhisper LoRA fine-tune — Kaggle runner

Config-driven pipeline. `src/` never hardcodes a Kaggle path — this notebook is the
only place `/kaggle/input/...` appears, passed in via `--override`.

**Before running**: attach as Kaggle Dataset inputs (Add Data):
- `paid-dataset-v2` (from `dataset/paid-dataset-v2/` in this repo, ~985 MB — consolidated
  2026-08-02 from the dot2 data drop + legacy test meetings repurposed as train, see
  `PROJECT_CORE.md` §4 and `scripts/ingest_paid_dataset_v2.py`. Zip and upload as a new
  Kaggle Dataset — this supersedes the old `paid-dataset` attachment.)
- `youtube-meetings` (from `dataset/youtube-meetings/` in this repo -- manifest + `audio/`
  only, **not** `raw/`, ~230 MB. Must be fully reviewed first: `scripts/review_youtube.py
  --check` must pass with no `verified: false` records. Merged with `paid-dataset-v2` by
  the new cell in step 2 below into `dataset/mixed-noisy-v1`, see
  `youtube-data-pilot/README.md` step 6.)
- `real-meetings-bench` (from `dataset/real-meetings-bench/` in this repo, ~80 MB,
  produced by `scripts/ingest_real_bench.py` — zip and upload as a Kaggle Dataset)
- (optional) a GPU accelerator (T4 x1 is enough — see handoff, batch 8 measured at 9.75 GiB
  for -small; -large needs its own batch size measured live, see Cell 4)

VIVOS (OOD) does not need a Kaggle Dataset attachment — `scripts/fetch_vivos.py`
downloads it directly from HF Hub in Cell 5.

Run cells **in order**, stopping to read output at each stage before continuing —
this pipeline has never run end-to-end on `mixed-noisy-v1`; do not queue all cells blind.

## 1. Clone / update the repo

In [1]:
import os

# Force single-GPU: on a T4 x2 session, Trainer/accelerate auto-wraps the model in
# legacy torch.nn.DataParallel when it sees >1 visible GPU without a distributed
# launch (accelerate launch / torchrun) -- that replicates the model and concentrates
# gradient reduction on one GPU, wasting memory for no speed benefit here. This
# pipeline is designed single-GPU only; set before any stage touches CUDA.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

REPO_URL = "https://github.com/egoist-minh/Reworkwhisper-finetune.git"
REPO_DIR = "/kaggle/working/Reworkwhisper-finetune"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
os.environ["TRANSFORMERS_AUTO_CONVERSION"] = "0"

Cloning into '/kaggle/working/Reworkwhisper-finetune'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 156 (delta 80), reused 120 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 213.06 KiB | 2.84 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/kaggle/working/Reworkwhisper-finetune


In [2]:
!pip install -q -r requirements.txt

## 2. Locate attached datasets

Confirm the exact mount paths before setting the overrides below — Kaggle slugs the
dataset name, so this can differ from what you expect.

In [3]:
!ls -la /kaggle/input

total 12
drwxr-xr-x 3 root root 4096 Aug  3 06:38 .
drwxr-xr-x 8 root root 4096 Aug  3 06:38 ..
drwxr-xr-x 3 root root 4096 Aug  3 06:38 datasets


In [ ]:
# Merge paid-dataset-v2 (synthetic) with youtube-meetings (real, reviewed YouTube
# audio -- scripts/review_youtube.py --check must already pass) into one directory
# so configs/experiment.yaml:data.dataset_path can stay a single path. Kaggle mount
# paths nest one level deeper than the attached dataset name (Cell above exists to
# confirm this) -- edit both --sources paths to match what it printed. Run as a
# module (-m scripts.build_mixed_dataset), not a bare file path -- it imports
# scripts.review_youtube, same convention as
# youtube-data-pilot/style-guide.md's `python -m scripts.review_youtube`.
!python -m scripts.build_mixed_dataset \
    --sources /kaggle/input/datasets/winhkento/paid-dataset-v2/paid-dataset-v2 \
              /kaggle/input/datasets/winhkento/youtube-meetings/youtube-meetings \
    --out /kaggle/working/dataset/mixed-noisy-v1

## 3. Set platform-specific paths

Edit these three to match what Cell above printed. This is the *only* place a
`/kaggle/input/...` path is written — everything downstream goes through
`--override`, never a hardcoded path inside `src/`.

In [ ]:
DATASET_PATH = "/kaggle/working/dataset/mixed-noisy-v1"       # written by the build_mixed_dataset.py cell above
REAL_BENCH_PATH = "/kaggle/input/datasets/winhkento/real-meetings-bench/real-meetings-bench"  # edit to match Cell 2's listing
OOD_EVAL_PATH = "/kaggle/working/Reworkwhisper-finetune/dataset/vivos"  # written by Cell 4

OVERRIDES = (
    f"--override data.dataset_path={DATASET_PATH} "
    f"--override data.real_bench_path={REAL_BENCH_PATH} "
    f"--override data.ood_eval_path={OOD_EVAL_PATH}"
)
print(OVERRIDES)

## 4. Model & hyperparameters

Every value below is a real field in `configs/experiment.yaml`, applied via
`--override` — this isn't a new config surface, just a convenient place to see and
change what a run actually uses instead of hand-editing YAML or writing override
strings from scratch.

**`EVAL_LIMIT` — read this before trusting any number from a prior run.**
`configs/experiment.yaml` now ships `eval.limit: null` (fixed 2026-08-02 — used to be
`20`, an early smoke-testing leftover). `src/gate.py`'s `_eval_split` applies this to
*every* eval — baseline **and** every sweep-gate tier. With `paid-dataset-v2`, the
full splits are test **426** (was 236 — now voice-disjoint from train, see
`PROJECT_CORE.md` §4) / VIVOS 760 / real-bench 264. `EVAL_LIMIT` below should stay
`null` for a trustworthy run.

In [5]:
BASE_MODEL = "vinai/PhoWhisper-large"    # 2026-08-03 deadline target. ~5.7x more params than
                                          # -small (1.64B vs 288M) -- no measured batch size for
                                          # -large in THIS repo yet; TRAIN_BATCH_SIZE below is a
                                          # conservative STARTING POINT, not a measured number --
                                          # watch actual GPU memory on the first run and adjust.

LORA_RANK = 16
LORA_ALPHA = 32

TRAIN_EPOCHS = 3
TRAIN_BATCH_SIZE = 2              # unmeasured starting point for -large (measured for -small only:
                                   # T4 x1 peak 9.75 GiB at batch 8, 4 -> 8.05, 2 -> 7.21). Raise only
                                   # after confirming this fits, watching nvidia-smi during the run.
GRAD_ACCUM_STEPS = 8              # raised to keep effective batch (16) close to the -small run's
LEARNING_RATE = 2.0e-4
TRAIN_LIMIT = "null"
EVAL_LIMIT ="null"# null = full split. See markdown above -- was left at 20 (smoke-test value)
EVAL_BATCH_SIZE = 8

SWEEP_LAMBDAS = "[0.0,0.25,0.5,0.75,1.0]"
OOD_CER_BUDGET = 0.02
REAL_CER_REGRESSION_PP = 0.0     # tier 4a zero-tolerance -- loosen only with a deliberate decision, see SESSIONS.md

PARAM_OVERRIDES = (
    f"--override base_model={BASE_MODEL} "
    f"--override lora.rank={LORA_RANK} "
    f"--override lora.alpha={LORA_ALPHA} "
    f"--override training.epochs={TRAIN_EPOCHS} "
    f"--override training.batch_size={TRAIN_BATCH_SIZE} "
    f"--override training.grad_accum_steps={GRAD_ACCUM_STEPS} "
    f"--override training.limit={TRAIN_LIMIT} "
    f"--override training.learning_rate={LEARNING_RATE} "
    f"--override eval.limit={EVAL_LIMIT} "
    f"--override eval.batch_size={EVAL_BATCH_SIZE} "
    f"--override 'sweep.lambdas={SWEEP_LAMBDAS}' "
    f"--override sweep.ood_cer_budget={OOD_CER_BUDGET} "
    f"--override gates.real_cer_regression_pp={REAL_CER_REGRESSION_PP}"
)

OVERRIDES = OVERRIDES + " " + PARAM_OVERRIDES
print(OVERRIDES)

--override data.dataset_path=/kaggle/input/datasets/winhkento/paid-dataset-v2/paid-dataset-v2 --override data.real_bench_path=/kaggle/input/datasets/winhkento/real-meetings-bench/real-meetings-bench --override data.ood_eval_path=/kaggle/working/Reworkwhisper-finetune/dataset/vivos --override base_model=vinai/PhoWhisper-large --override lora.rank=16 --override lora.alpha=32 --override training.epochs=3 --override training.batch_size=2 --override training.grad_accum_steps=8 --override training.limit=null --override training.learning_rate=0.0002 --override eval.limit=null --override eval.batch_size=8 --override 'sweep.lambdas=[0.0,0.25,0.5,0.75,1.0]' --override sweep.ood_cer_budget=0.02 --override gates.real_cer_regression_pp=0.0


## 5. Fetch VIVOS (OOD benchmark)

**Untested end-to-end before this run** — parquet route primary, tarball fallback.
Read the printed schema before trusting the manifest it writes.

In [6]:
!python scripts/fetch_vivos.py --out dataset/vivos --smoke

default/test/0000.parquet: 100%|███████████| 85.0M/85.0M [00:03<00:00, 23.4MB/s]
parquet route: wrote 5 segments
manifest: dataset/vivos/manifest.vivos.jsonl


In [7]:
# If the smoke run above looks right, fetch the full test split (no --smoke / --limit):
!python scripts/fetch_vivos.py --out dataset/vivos

parquet route: wrote 760 segments
manifest: dataset/vivos/manifest.vivos.jsonl


## 6. Stage: smoke

CPU-only, no model download. Proves config load, manifest merge, split resolution,
normalization, and the peft compat patch all work on this exact Kaggle image before
any GPU time is spent. **This has never run on Kaggle before — read the output
carefully, do not assume it just works.**

In [8]:
!python -m src.pipeline --stage smoke {OVERRIDES}

compat: []
split_stats: {'train': 4270, 'val': 250, 'test': 426}
normalize sample: Rồi, mọi người vào họp thôi, mình cần quyết nhanh vụ này trước khi đến giờ deploy chiều nay. -> rồi mọi người vào họp thôi mình cần quyết nhanh vụ này trước khi đến giờ deploy chiều nay
SMOKE OK


## 7. Stage: baseline

Base model over test + OOD + real bench. Writes `metrics/baseline.json` and
`audit/predictions_baseline_*.csv`. **Only run this after Cell 6 (smoke) is clean.**

In [ ]:
RUN_ID = "v4-mixed-r16"  # matches configs/experiment.yaml:run_id unless overridden here
!TRANSFORMERS_AUTO_CONVERSION=0 python -m src.pipeline --stage baseline --override run_id={RUN_ID} {OVERRIDES}

In [10]:
import json
print(json.dumps(json.load(open(f"outputs/{RUN_ID}/metrics/baseline.json")), indent=2))

{
  "cer_test": 0.04259333725138421,
  "cer_ood": 0.022832927484899276,
  "cer_real": 0.4575888272745944
}


## 8. Stage: train

LoRA SFT, rank from `configs/experiment.yaml` (16, or `LORA_RANK` above if you
changed it). Two things unverified on real GPU as of 2026-08-02, watch the first
few log lines closely before letting either run the full epochs:
- `Trainer(eval_dataset=dict)` multi-eval-set API (previously exercised successfully
  on -small, per `SESSIONS.md`, but never yet on `paid-dataset-v2`).
- Early stopping AND best-checkpoint selection were just replaced with one custom
  callback (`src/train.py:_EarlyStoppingState` / `RobustEvalTrackingCallback`,
  unit-tested locally but never run against real transformers) — it no longer uses
  the built-in `EarlyStoppingCallback` or `load_best_model_at_end`/
  `metric_for_best_model` at all (both depended on the same fragile metric-key
  lookup). It saves `checkpoints/best/` itself the instant `eval_val_cer` improves,
  and raises loudly if `eval_val_cer` is never observed in any eval round rather
  than silently shipping an empty/wrong checkpoint. Confirm `checkpoints/best/`
  actually gets written partway through training, not only (or never) at the end.

In [11]:
!python -m src.pipeline --stage train --override run_id={RUN_ID} {OVERRIDES}

Loading weights: 100%|█| 1260/1260 [00:00<00:00, 2508.63it/s, Materializing para
trainable params: 28,835,840 || all params: 1,638,528,000 || trainable%: 1.7599
  Epoch    Step   TrainLoss   ValLoss   ValCER   ValWER   OOD_CER
    1.0     267       0.643     0.087   0.0119   0.0277    0.0559
    2.0     534       0.258     0.066   0.0105   0.0228    0.0494
    3.0     801       0.047     0.075   0.0101   0.0233    0.0420
train: 100%|██████████████████| 801/801 [6:31:39<00:00, 29.34s/step, loss=0.047]


## 9. HF token (only needed if you intend to push in Cell 10)

Add `HF_TOKEN` under this notebook's Add-ons → Secrets first. Never hardcode the
token here — it must not end up in any committed artifact.

Move/run this cell earlier (right after Cell 2) if you want it set for every stage
— that also silences the "unauthenticated requests to the HF Hub" warning that
otherwise appears on every stage's model load.

In [12]:
from kaggle_secrets import UserSecretsClient
import os

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print(f"No HF_TOKEN secret configured ({e}) -- fine if hub.push is false")

HF_TOKEN set


## 10. Stage: sweep-gate

λ sweep (hard-fails if no λ fits `sweep.ood_cer_budget` — no fallback) → gate tiers
1/2/4a → HF push iff `overall_pass` and `hub.push: true`. Set `hub.push`/`hub.repo_id`
below only once you've decided to actually publish — this is an outward-facing action.

**Tier 4a now reports more than pass/fail (added 2026-08-02)** — `gate_results.json`'s
`tier4a_real` carries `by_meeting` (CER per real recording, don't just read the pooled
number), `delta_ci`/`verdict` (paired comparison vs baseline on the same segments —
read `verdict`: `INCONCLUSIVE` means the ~264-segment sample can't resolve the
difference, treat that as "no evidence," not as a pass), and `normalization_check`
(if the two number-convention CERs differ a lot, the result is normalization-driven,
not model-driven). None of these change `pass`/`overall_pass` — read them alongside
the gate verdict, not instead of it.

In [13]:
HUB_PUSH = False        # flip to True only when ready to publish
HUB_REPO_ID = None       # e.g. "your-username/phowhisper-lora-v0-r16"

hub_overrides = f"--override hub.push={HUB_PUSH} " + (f"--override hub.repo_id={HUB_REPO_ID} " if HUB_REPO_ID else "")
!python -m src.pipeline --stage sweep-gate --override run_id={RUN_ID} {OVERRIDES} {hub_overrides}

Loading weights: 100%|█| 1260/1260 [00:01<00:00, 693.71it/s, Materializing param
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [14]:
import json
print(json.dumps(json.load(open(f"outputs/{RUN_ID}/metrics/gate_results.json")), indent=2))

{
  "tier1_in_domain": {
    "cer": 0.01710538525781806,
    "bound": 0.03833400352624579,
    "pass": true
  },
  "tier2_ood": {
    "cer": 0.041428164827171814,
    "bound": 0.04283292748489928,
    "pass": true
  },
  "tier4a_real": {
    "cer": 0.480078044773054,
    "ci": [
      0.40152576677565,
      0.565982972136223
    ],
    "bound": 0.4575888272745944,
    "pass": false,
    "by_meeting": {
      "real_0001": {
        "cer": 0.452259802454355,
        "wer": 0.5313267813267813,
        "n_segments": 146
      },
      "real_0002": {
        "cer": 0.5035331230283911,
        "wer": 0.5560133917074427,
        "n_segments": 118
      }
    },
    "delta_ci": [
      -0.06257205743632743,
      0.029187769649221484
    ],
    "verdict": "INCONCLUSIVE",
    "normalization_check": {
      "word_to_digit": 0.480078044773054,
      "as_written": 0.4852127262232182,
      "delta_pp": 0.513
    }
  },
  "overall_pass": false
}


In [ ]:
# On v3-r16, tqdm's redraw overwrote the last printed line, so the sweep table and
# the selected lambda* never showed up anywhere in that notebook -- only in the CSV.
# Print it here so this run's record isn't the same blind spot.
import csv
with open(f"outputs/{RUN_ID}/metrics/lambda_sweep.csv") as f:
    for row in csv.DictReader(f):
        print(row)

## 11. Evidence — CER + predictions

Everything under `outputs/{run_id}/` is the run's evidence: `metrics/baseline.json`,
`metrics/lambda_sweep.csv`, `metrics/gate_results.json`, and every
`audit/predictions_*.csv` (segment-level ref/hyp for baseline and gate, per tier).
Download this whole folder before the Kaggle session ends — it is not saved anywhere
else.

In [15]:
!find outputs/{RUN_ID} -type f | sort

outputs/v0-r16/adapter/adapter_config.json
outputs/v0-r16/adapter/adapter_model.safetensors
outputs/v0-r16/adapter/README.md
outputs/v0-r16/audit/predictions_baseline_ood.csv
outputs/v0-r16/audit/predictions_baseline_real.csv
outputs/v0-r16/audit/predictions_baseline_test.csv
outputs/v0-r16/audit/predictions_tier1_in_domain.csv
outputs/v0-r16/audit/predictions_tier2_ood.csv
outputs/v0-r16/audit/predictions_tier4a_real.csv
outputs/v0-r16/checkpoints/best/adapter_config.json
outputs/v0-r16/checkpoints/best/adapter_model.safetensors
outputs/v0-r16/checkpoints/best/README.md
outputs/v0-r16/checkpoints/checkpoint-267/adapter_config.json
outputs/v0-r16/checkpoints/checkpoint-267/adapter_model.safetensors
outputs/v0-r16/checkpoints/checkpoint-267/optimizer.pt
outputs/v0-r16/checkpoints/checkpoint-267/README.md
outputs/v0-r16/checkpoints/checkpoint-267/rng_state.pth
outputs/v0-r16/checkpoints/checkpoint-267/scaler.pt
outputs/v0-r16/checkpoints/checkpoint-267/scheduler.pt
outputs/v0-r16/checkpo

In [16]:
!zip -r -q outputs_{RUN_ID}.zip outputs/{RUN_ID}

from IPython.display import FileLink
FileLink(f"outputs_{RUN_ID}.zip")

/kaggle/working/Reworkwhisper-finetune/outputs_v0-r16.zip